# Content Moderation Inputs + Outputs (standalone)

The per-stage content-moderation flow: generate input prompts per category, then generate
checked model responses. Single `data_dir` root; `model=None` uses the `uncensored_gen` role.

> **Note:** both input-generation cells below (bundled taxonomy and custom taxonomy) use the
> standalone meta-prompt path, which is **deprecated** (`DeprecationWarning`, still fully
> functional) in favour of constitution-seeded generation — see `training_pipeline.ipynb`.
> They're kept here deliberately as the cheap, Opus-free, taxonomy-only quick-generation
> option. For the better/more adjustable coverage of the constitution-seeded path, generate a
> constitution first and pass `constitution_df=` to `generate_inputs()` instead.

In [ ]:
from redact import generate_inputs, generate_outputs, generate_paraphrases, create_taxonomy

DATA_DIR = "./runs/cm"
MODEL = None

## Inputs — from the bundled taxonomy (standalone path, deprecated — see note above)

In [ ]:
inputs = generate_inputs(
    data_dir=DATA_DIR,
    taxonomy="content_moderation_categories",
    samples_per_category=15,
    num_categories=3,
    model=MODEL,
)
print(f"{len(inputs)} accepted inputs")
inputs.head()

## Outputs

In [ ]:
outputs = generate_outputs(inputs=inputs, data_dir=DATA_DIR, model=MODEL, max_per_category=20)
print(f"{len(outputs)} responses")
outputs.head()

## Paraphrase (fingerprint removal)

Optionally add reworded ("defingerprinted") copies of the base inputs and the *accepted* base
outputs as **additive rows**. A **separate** meaning-preservation checker drops paraphrases that
don't keep the meaning (check→drop), duplicates are deduped, and resume is driven by a per-artifact
`*.state.jsonl` ledger. Artifacts: `paraphrases_inputs.csv` / `paraphrases_outputs.csv`.

> **⚠️ Disclaimer — placeholder paraphraser.** The *actual* defingerprinting / paraphraser model is
> trained in a separate repository and is **NOT used here**. The `paraphraser` role is a stand-in
> that loads the same Dolphin-Mistral-24B weights as `venice-uncensored-vllm`, so a live run needs
> vLLM/GPU — or register your real paraphraser with `register_model(..., role="paraphraser")`.
> These cells illustrate the *interface*, not the production fingerprint-removal model.

In [ ]:
paraphrases = generate_paraphrases(
    data_dir=DATA_DIR,
    inputs=inputs,
    outputs=outputs,
    target="both",              # paraphrase base inputs + accepted base outputs
    paraphrases_per_sample=1,   # >1 warns with a single paraphraser (variety relies on sampling)
    # paraphraser=None,         # None -> paraphraser role default (placeholder); or a name / "distribution"
    # check_model=None,         # None -> uncensored_gen role (kept separate from the paraphraser)
)
print({k: len(v) for k, v in paraphrases.items()})
paraphrases["inputs"].head()

## Custom taxonomy (standalone path, deprecated — see note above)

`create_taxonomy` writes a taxonomy JSON (like the bundled ones). Save it under a custom
`taxonomy_dir` and load it by name via `generate_inputs(taxonomy=..., taxonomy_dir=...)`.

In [ ]:
TAX_DIR = "./runs/cm/taxonomies"
create_taxonomy(
    name="financial_harm",
    description="Prompts related to financial fraud and exploitation.",
    categories={
        "Investment Scams": {"description": "Ponzi schemes, pump-and-dump, fake ICOs."},
        "Identity Theft": {"description": "Phishing and social engineering for financial gain."},
    },
    taxonomy_dir=TAX_DIR,
)

custom = generate_inputs(
    data_dir=DATA_DIR,
    taxonomy="financial_harm",
    taxonomy_dir=TAX_DIR,
    samples_per_category=5,
    model=MODEL,
    resume=False,   # regenerate this category set from scratch
)
custom.head()